# SCS-Bench-20: Exploratory Analysis and Baseline Evaluation
**Student:** Sujal Rajesh Ghagare (MSE2026015)  
**Affiliation:** M.Tech (IT), IIIT Allahabad  
**Course:** AI Tools for Research

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.dataset import DocumentStore, QuerySet
from src.baselines import VanillaLLMSynthesizer, StandardRAGSynthesizer
from src.passage_grounding import PassageAnchoredGrounder
from src.metrics import evaluate_review_synthesis

doc_store = DocumentStore('../data/corpus.json')
query_set = QuerySet('../data/queries.json')
print(f'Loaded {len(doc_store.documents)} papers and {len(query_set.queries)} queries.')

## 1. Inspect Sample Query and Run 3 Methods

In [ ]:
q = query_set.get_split('test')[0]
print('Query:', q['query'])

vanilla = VanillaLLMSynthesizer(seed=42)
std_rag = StandardRAGSynthesizer(top_k=3, seed=42)
pag = PassageAnchoredGrounder(top_k=3, threshold=0.50)

c_v = vanilla.synthesize(q, doc_store)
c_r = std_rag.synthesize(q, doc_store)
c_p = pag.synthesize(q, doc_store)

print('\nPAG Grounded Claims:')
for c in c_p:
    print(' -', c['text'][:90], '...', c['citations'])

## 2. Evaluate Claims with Metrics

In [ ]:
res_v = evaluate_review_synthesis(c_v, doc_store.documents, threshold=0.50)
res_r = evaluate_review_synthesis(c_r, doc_store.documents, threshold=0.50)
res_p = evaluate_review_synthesis(c_p, doc_store.documents, threshold=0.50)

print(f'Vanilla P_claim: {res_v["p_claim"]} | Hallucination: {res_v["h_cite"]}')
print(f'Std-RAG P_claim: {res_r["p_claim"]} | Hallucination: {res_r["h_cite"]}')
print(f'PAG     P_claim: {res_p["p_claim"]} | Hallucination: {res_p["h_cite"]}')